# HistGradientBoosting + Exact-Value Target Encoding

An approach built on top of our CatBoost baseline, testing two ingredients adapted
from [TE-HGBC](https://www.kaggle.com/code/redamountassir/ps-s6e7-hgbc-baseline-lb-0-95034-cv-0-95026)
by redamountassir:

1. **Exact-value target encoding of numeric features** -- all 13 raw columns
   (6 categoricals + 7 numerics) target-encoded via sklearn's native
   `TargetEncoder(cv=5, target_type='multiclass')`, with numerics cast to exact
   string values rather than binned. With 690k rows, each distinct numeric
   value repeats hundreds of times, so per-value label rates directly estimate
   the generator's probability -- signal `HistGradientBoostingClassifier`'s <=255
   histogram bins can't fully resolve on their own.
2. **`HistGradientBoostingClassifier`** (scikit-learn's native gradient boosting
   implementation), with native `class_weight='balanced'` and native categorical
   handling.

We reuse the source notebook's tuned hyperparameters as-is (this experiment is
about testing the TE+HGBC combination, not re-tuning capacity), use a single
5-fold stratified split, and apply no post-hoc prior/threshold correction --
plain argmax, since stacking training-time class weighting with a separate
post-hoc correction is a well-documented pitfall on this competition (multiple
public notebooks independently found it hurts rather than helps).

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='catboost')

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import TargetEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/playground-series-s6e7')
if not (DATA_DIR / 'train.csv').exists():
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(hits[0]).parent
    else:
        DATA_DIR = Path('..') / 'data'  # local fallback for testing outside Kaggle
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
print('DATA_DIR:', DATA_DIR)

TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']
BASE_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS
RAW_COLS = NUMERIC_COLS + CATEGORICAL_COLS

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

for col in CATEGORICAL_COLS:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))
print('train', train.shape, 'test', test.shape)

sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

## Two feature views

**Model view**: native `pandas.Categorical` dtype for categoricals (shared
categories, NaN as its own level), numerics left as-is (HGBC natively routes
NaN, no imputation needed).

**TE view**: all 13 raw columns cast to string (numerics at exact value), NaN
filled with a placeholder — this is what gets target-encoded per fold.

In [ ]:
train_model = train.copy()
test_model = test.copy()
for col in CATEGORICAL_COLS:
    categories = sorted(set(train_model[col].unique()) | set(test_model[col].unique()))
    train_model[col] = pd.Categorical(train_model[col], categories=categories)
    test_model[col] = pd.Categorical(test_model[col], categories=categories)

train_te_str = train[RAW_COLS].astype(str).fillna('na')
test_te_str = test[RAW_COLS].astype(str).fillna('na')
print('TE-view cardinalities:')
print(train_te_str.nunique().sort_values(ascending=False).to_string())

## Member A — HistGradientBoosting + exact-value target encoding

Hyperparameters reused as-is from the source notebook's tuned operating point.
`TargetEncoder(cv=5)` is fit fresh per outer fold on that fold's training rows
only (leak-free via internal cross-fitting), producing `13 * 3 = 39` TE
columns concatenated onto the model-view features.

In [ ]:
HGBC_CONFIG = dict(
    learning_rate=0.0627037115235577,
    max_iter=300,
    max_leaf_nodes=33,
    min_samples_leaf=298,
    l2_regularization=0.028912644384523085,
    max_bins=237,
    max_features=0.820265066682815,
    early_stopping=True,
    categorical_features='from_dtype',
    class_weight='balanced',
    random_state=42,
)

te_names = [f'te_{c}_{k}' for c in RAW_COLS for k in range(n_classes)]
oof_proba_hgbc = np.zeros((len(train_model), n_classes))
test_proba_hgbc = np.zeros((len(test_model), n_classes))

for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc='HGBC-TE folds')):
    enc = TargetEncoder(cv=5, smooth='auto', target_type='multiclass')
    te_tr = enc.fit_transform(train_te_str.iloc[tr_idx], y[tr_idx])
    te_val = enc.transform(train_te_str.iloc[val_idx])
    te_test = enc.transform(test_te_str)

    X_tr = pd.concat([train_model[BASE_FEATURES].iloc[tr_idx].reset_index(drop=True),
                       pd.DataFrame(te_tr, columns=te_names)], axis=1)
    X_val = pd.concat([train_model[BASE_FEATURES].iloc[val_idx].reset_index(drop=True),
                        pd.DataFrame(te_val, columns=te_names)], axis=1)
    X_test = pd.concat([test_model[BASE_FEATURES].reset_index(drop=True),
                         pd.DataFrame(te_test, columns=te_names)], axis=1)
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = HistGradientBoostingClassifier(**HGBC_CONFIG)
    model.fit(X_tr, y_tr)
    oof_proba_hgbc[val_idx] = model.predict_proba(X_val)
    val_pred = oof_proba_hgbc[val_idx].argmax(axis=1)
    fold_score = balanced_accuracy_score(y_val, val_pred)
    print(f'fold {fold}: balanced accuracy = {fold_score:.5f} (n_iter={model.n_iter_})')
    test_proba_hgbc += model.predict_proba(X_test) / N_FOLDS

oof_pred_hgbc = oof_proba_hgbc.argmax(axis=1)
oof_bal_acc_hgbc = balanced_accuracy_score(y, oof_pred_hgbc)
print(f'\nHGBC-TE OOF balanced accuracy: {oof_bal_acc_hgbc:.5f}')
print('source notebook reported CV (different fold split): 0.95026')
print()
print(classification_report(y, oof_pred_hgbc, target_names=label_encoder.classes_, digits=4))

## Member B — CatBoost-V1 (reproducing v0.3's exact config, base features)

Same-data comparison peg (current best single model) and a blend check.

In [ ]:
class BalancedAccuracyMetric:
    def is_max_optimal(self):
        return True
    def evaluate(self, approxes, target, weight):
        approx = np.array(approxes)
        pred = approx.argmax(axis=0)
        y_true = np.array(target).astype(int)
        return balanced_accuracy_score(y_true, pred), 1.0
    def get_final_error(self, error, weight):
        return error

oof_proba_cb = np.zeros((len(train), n_classes))
test_proba_cb = np.zeros((len(test), n_classes))
cb_best_iters = []

for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc='CatBoost-V1 folds')):
    X_tr, X_val = train[BASE_FEATURES].iloc[tr_idx], train[BASE_FEATURES].iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = CatBoostClassifier(
        loss_function='MultiClass', eval_metric=BalancedAccuracyMetric(),
        auto_class_weights='Balanced', iterations=3000, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, random_seed=42, task_type='CPU', verbose=False,
    )
    model.fit(X_tr, y_tr, cat_features=CATEGORICAL_COLS, eval_set=(X_val, y_val),
              early_stopping_rounds=100)
    oof_proba_cb[val_idx] = model.predict_proba(X_val)
    cb_best_iters.append(model.best_iteration_)
    test_proba_cb += model.predict_proba(test[BASE_FEATURES]) / N_FOLDS

oof_pred_cb = oof_proba_cb.argmax(axis=1)
oof_bal_acc_cb = balanced_accuracy_score(y, oof_pred_cb)
print('best_iterations:', cb_best_iters)
print(f'CatBoost-V1 OOF balanced accuracy: {oof_bal_acc_cb:.4f}')
print('v0.3 Variant 1 known result:       0.9493, best_iterations [428, 950, 605, 339, 779]')

In [ ]:
assert abs(oof_bal_acc_cb - 0.9493) < 0.002, 'CatBoost-V1 reproduction did not match v0.3 Variant 1 closely enough'
print('CatBoost-V1 reproduction check: PASS')

## Solo comparison

In [ ]:
solo_scores = {'hgbc_te': oof_bal_acc_hgbc, 'catboost_v1': oof_bal_acc_cb}
for name, score in sorted(solo_scores.items(), key=lambda kv: -kv[1]):
    print(f'{score:.4f}  {name}')

## Blend weight search + nested validation

In [ ]:
OOF_PROBAS = {'hgbc_te': oof_proba_hgbc, 'catboost_v1': oof_proba_cb}
TEST_PROBAS = {'hgbc_te': test_proba_hgbc, 'catboost_v1': test_proba_cb}
MEMBER_NAMES = ['hgbc_te', 'catboost_v1']

def pair_grid(step=0.02):
    n_steps = round(1 / step)
    return [(i * step, (n_steps - i) * step) for i in range(n_steps + 1)]

def blend_predict(probas_dict, weights, members):
    blended = sum(w * probas_dict[m] for w, m in zip(weights, members))
    return blended.argmax(axis=1)

def grid_search_blend(oof_probas_dict, y_true, members, grid):
    best_score, best_w = -1, tuple(1 / len(members) for _ in members)
    for w in tqdm(grid, desc=f'blend grid ({"+".join(members)})', leave=False):
        pred = blend_predict(oof_probas_dict, w, members)
        score = balanced_accuracy_score(y_true, pred)
        if score > best_score:
            best_score, best_w = score, w
    return best_score, best_w

grid2 = pair_grid(step=0.02)
best_score_2way, best_w_2way = grid_search_blend(OOF_PROBAS, y, MEMBER_NAMES, grid2)
print(f'2-way blend full-OOF grid search: best {best_score_2way:.4f} at weights {dict(zip(MEMBER_NAMES, (round(x,2) for x in best_w_2way)))}')
print(f'Best solo model:                  {max(solo_scores.values()):.4f}')
print(f'Improvement (same-data, likely optimistic): {best_score_2way - max(solo_scores.values()):+.4f}')

In [ ]:
nested_scores_solo_best = []
nested_scores_blend = []
nested_weights_2way = []
best_solo_key = max(solo_scores, key=solo_scores.get)

for fold_idx, (_, val_idx) in enumerate(tqdm(folds, desc='nested CV (2-way blend)')):
    other_idx = np.concatenate([folds[j][1] for j in range(N_FOLDS) if j != fold_idx])
    other_oof = {m: OOF_PROBAS[m][other_idx] for m in MEMBER_NAMES}
    _, w_fold = grid_search_blend(other_oof, y[other_idx], MEMBER_NAMES, grid2)

    val_oof = {m: OOF_PROBAS[m][val_idx] for m in MEMBER_NAMES}
    blend_pred = blend_predict(val_oof, w_fold, MEMBER_NAMES)
    solo_pred = OOF_PROBAS[best_solo_key][val_idx].argmax(axis=1)

    nested_scores_solo_best.append(balanced_accuracy_score(y[val_idx], solo_pred))
    nested_scores_blend.append(balanced_accuracy_score(y[val_idx], blend_pred))
    nested_weights_2way.append(w_fold)

print('Per-fold blend weights (fit on the other 4 folds):', [tuple(round(x, 2) for x in w) for w in nested_weights_2way])
print(f'Nested best-solo-model ({best_solo_key}) balanced accuracy: {np.mean(nested_scores_solo_best):.4f} (+/- {np.std(nested_scores_solo_best):.4f})')
print(f'Nested 2-way-blend balanced accuracy:                      {np.mean(nested_scores_blend):.4f} (+/- {np.std(nested_scores_blend):.4f})')
print(f'Honest improvement estimate: {np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best):+.4f}')

## Decision + candidate submission

In [ ]:
honest_improvement = np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best)
THRESHOLD_FOR_REAL_IMPROVEMENT = 0.0005
CURRENT_BEST_OOF = 0.9493

print(f'HGBC-TE solo:              {oof_bal_acc_hgbc:.4f}')
print(f'CatBoost-V1 solo:          {oof_bal_acc_cb:.4f}')
print(f'Current project best OOF: {CURRENT_BEST_OOF:.4f}')
print()

if honest_improvement > THRESHOLD_FOR_REAL_IMPROVEMENT and (np.mean(nested_scores_blend) > CURRENT_BEST_OOF + THRESHOLD_FOR_REAL_IMPROVEMENT):
    print(f'REAL IMPROVEMENT: blend {honest_improvement:+.4f} over solo (nested-validated), and beats current best. Using weights fit on full OOF: {dict(zip(MEMBER_NAMES, best_w_2way))}')
    test_pred = blend_predict(TEST_PROBAS, best_w_2way, MEMBER_NAMES)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
    assert submission[TARGET].isnull().sum() == 0
    submission_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows)')
    display(submission[TARGET].value_counts(normalize=True))
elif oof_bal_acc_hgbc > CURRENT_BEST_OOF + THRESHOLD_FOR_REAL_IMPROVEMENT:
    print(f'HGBC-TE solo beats the current best by {oof_bal_acc_hgbc - CURRENT_BEST_OOF:+.4f} -- worth a submission on its own merits.')
    test_pred = test_proba_hgbc.argmax(axis=1)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
    assert submission[TARGET].isnull().sum() == 0
    submission_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows)')
    display(submission[TARGET].value_counts(normalize=True))
else:
    print(f'NO REAL IMPROVEMENT over the current best (~{CURRENT_BEST_OOF:.4f} OOF). '
          f'Not writing a new submission.csv.')

## Summary

**Positive result — new best model.** Run on Kaggle's own compute (~86 minutes
total; HGBC-TE itself finished in under 5 minutes, the CatBoost-V1 reproduction
peg took the bulk of the time).

- **HGBC-TE solo OOF: 0.9502** — closely matches the source notebook's own
  reported CV (0.95026) despite a different fold split, and beats CatBoost-V1
  (0.9493) by **+0.0009**, the first genuine, non-noise-level improvement across
  this project's entire squeeze phase (every prior attempt landed within ±0.0005
  of CatBoost-V1). Per-class recall: at-risk 0.9373, fit 0.9500, unhealthy 0.9633.
- CatBoost-V1 reproduction: **PASS** (exact match, `best_iterations [428, 950,
  605, 339, 779]`, OOF 0.9493).
- Blend check: full-OOF 2-way grid search best 0.9504 at weights (hgbc_te=0.78,
  catboost_v1=0.22) — only +0.0002 over solo. Nested-validated honest
  improvement for the blend: **+0.0002**, below the 0.0005 threshold. HGBC-TE's
  solo score independently clears `current_best + 0.0005` on its own, so the
  decision logic correctly submitted the solo HGBC-TE predictions rather than
  the marginally-better-but-not-worth-it blend.
- **Submitted to Kaggle: public LB 0.95036** vs. OOF 0.9502 — tight correlation,
  no haircut. New best LB in this project.

This revises the "synthesis-noise ceiling" conclusion from Rungs 3-6: the
ceiling wasn't really at ~0.949 — a sufficiently different feature
representation (exact-value numeric target encoding, not binned/qcut'd) found
real additional signal that class-weighting, ensembling, one-vs-rest
decomposition, and threshold tuning had all missed. HGBC-TE is now the best
model in this project, superseding v0.3 CatBoost.